# 🕵️ Next-Token Detective - In-Class Guided Exercise

Investigate how a causal language model changes its next-token predictions when the context changes.

This notebook is a standalone companion to the **Model Inner Workings** section of `short_introduction_to_LLMs.ipynb`. The classroom notebook opened the model, examined its logits, applied softmax, and introduced `get_next_word_probabilities()`.

This exercise keeps the same idea but deliberately calls the result a **next-token** distribution. A token may be a word, part of a word, punctuation, whitespace, or another text fragment.


## Why This Exercise Matters


An LLM does not first decide on a complete answer and then print it. At each generation step, it produces scores for possible next tokens. The surrounding context reshapes those scores.

The purpose is **not to practice PyTorch**. Model loading, tensor operations, softmax, token decoding, and result formatting are supplied. Your work is to predict, experiment, compare, and explain.


## Exercise Scope


The required exercise contains three case files:

1. **Prediction lineup:** predict likely next tokens before revealing the model output.
2. **Context flips:** compare prompt pairs whose meanings change after a small context change.
3. **Create a case:** design and explain your own minimal pair.

A short optional task examines how concentrated a next-token distribution is.

**Suggested time:** 25–35 minutes for the required work, plus 5–10 minutes for the optional task.


## Learning Objectives


By the end of the exercise, you should be able to:

- explain why next-token predictions depend strongly on context,
- distinguish tokens from words,
- inspect several likely continuations instead of only the top candidate,
- recognize that a probability distribution is not a factual-truth score,
- design a small experiment that probes model behavior.


## 1. Environment Setup

Run the installation cell once when the required packages are not already available.

If you previously ran the classroom notebook in the same environment, the model files may already be cached.


In [ ]:
# !pip install -q "transformers>=4.51" accelerate pandas

In [ ]:
from typing import Any

import pandas as pd
import torch
from IPython.display import display
from transformers import AutoModelForCausalLM, AutoTokenizer

## 2. Model and Hardware Configuration

CUDA uses the same model as the classroom notebook. CPU-only environments use the smaller Qwen3 model by default, with SmolLM2 available as a lower-memory alternative.

Exact candidate tokens and probabilities may vary across models, dtypes, and library versions. The important evidence is the pattern produced by your own run.


In [ ]:
GPU_MODEL_ID = "unsloth/Llama-3.2-1B-Instruct"
CPU_MODEL_ID = "Qwen/Qwen3-0.6B"
LOW_MEMORY_CPU_MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"

# Leave as None for automatic detection. Set to "cuda" or "cpu" to override.
FORCE_DEVICE: str | None = None

# Enable only when the default CPU model is too demanding.
USE_LOW_MEMORY_CPU_MODEL = False

# "auto" is normally appropriate. torch.float32 is a compatibility fallback.
CPU_DTYPE: str | torch.dtype = "auto"

## 3. Supplied Model Loading

The following functions isolate hardware and model selection from the exercise. Read them, but do not treat them as student implementation tasks.


In [ ]:
def select_device_type() -> str:
    
    if FORCE_DEVICE is not None:
        requested = FORCE_DEVICE.lower()

        if requested not in {"cuda", "cpu"}:
            raise ValueError("FORCE_DEVICE must be None, 'cuda', or 'cpu'.")

        if requested == "cuda" and not torch.cuda.is_available():
            raise RuntimeError("CUDA was requested, but PyTorch cannot access a CUDA GPU.")

        return requested

    return "cuda" if torch.cuda.is_available() else "cpu"


def select_model_id(device_type: str) -> str:
    
    if device_type == "cuda":
        return GPU_MODEL_ID

    if USE_LOW_MEMORY_CPU_MODEL:
        return LOW_MEMORY_CPU_MODEL_ID

    return CPU_MODEL_ID


def load_runtime() -> dict[str, Any]:
    
    device_type = select_device_type()
    model_id = select_model_id(device_type)

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    if device_type == "cuda":
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            dtype=dtype,
            device_map="auto",
            low_cpu_mem_usage=True,
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            dtype=CPU_DTYPE,
            device_map="cpu",
            low_cpu_mem_usage=True,
        )

    model.eval()

    return {
        "model_id": model_id,
        "model": model,
        "tokenizer": tokenizer,
        "device_type": device_type,
        "device": next(model.parameters()).device,
    }

### Load the runtime


The first run may download the selected model. CPU inference will be slower, but this exercise uses only short single-step model calls.


In [ ]:
runtime = load_runtime()

print("Model:", runtime["model_id"])
print("Device:", runtime["device"])

## 4. Supplied Next-Token Tools

The classroom helper returned decoded strings. This version also exposes the token ID and the tokenizer's raw token piece so that whitespace and subword behavior remain visible.


In [ ]:
def make_whitespace_visible(text: str) -> str:
    """Make otherwise invisible whitespace easier to inspect."""

    return (
        text
        .replace(" ", "␠")
        .replace("\n", "↵")
        .replace("\t", "⇥")
    )

In [ ]:
def get_next_token_candidates(
    prompt: str,
    runtime: dict[str, Any],
    top_k: int = 10,
) -> pd.DataFrame:
    """Return the model's top-k next-token candidates for one prompt."""

    if not prompt:
        raise ValueError("prompt cannot be empty.")

    if top_k <= 0:
        raise ValueError("top_k must be positive.")

    tokenizer = runtime["tokenizer"]
    model = runtime["model"]

    encoded = tokenizer(prompt, return_tensors="pt").to(runtime["device"])

    with torch.inference_mode():
        output = model(**encoded)

    next_token_logits = output.logits[0, -1]
    probabilities = torch.softmax(next_token_logits.float(), dim=-1)

    k = min(top_k, probabilities.shape[-1])
    top_probabilities, top_token_ids = torch.topk(probabilities, k=k)

    rows = []

    for rank, (token_id, probability) in enumerate(
        zip(top_token_ids.tolist(), top_probabilities.tolist()),
        start=1,
    ):
        decoded = tokenizer.decode(
            [token_id],
            clean_up_tokenization_spaces=False,
        )
        token_piece = tokenizer.convert_ids_to_tokens(token_id)

        rows.append({
            "rank": rank,
            "token_id": token_id,
            "token_piece": token_piece,
            "decoded": decoded,
            "visible_decoded": make_whitespace_visible(decoded),
            "probability": probability,
        })

    return pd.DataFrame(rows)

In [ ]:
def show_next_token_candidates(
    prompt: str,
    runtime: dict[str, Any],
    top_k: int = 10,
) -> pd.DataFrame:
    """Display and return the top-k candidates for one prompt."""

    print(f"Prompt: {prompt!r}")
    candidates = get_next_token_candidates(prompt, runtime, top_k=top_k)

    display(
        candidates[
            [
                "rank",
                "token_id",
                "token_piece",
                "visible_decoded",
                "probability",
            ]
        ].style.format({"probability": "{:.4%}"})
    )

    return candidates

In [ ]:
def compare_prompt_pair(
    prompt_a: str,
    prompt_b: str,
    runtime: dict[str, Any],
    top_k: int = 6,
) -> pd.DataFrame:
    """Place two next-token candidate lists side by side."""

    candidates_a = get_next_token_candidates(prompt_a, runtime, top_k=top_k)
    candidates_b = get_next_token_candidates(prompt_b, runtime, top_k=top_k)

    comparison = pd.DataFrame({
        "rank": range(1, top_k + 1),
        "prompt_a_token": candidates_a["visible_decoded"],
        "prompt_a_probability": candidates_a["probability"],
        "prompt_b_token": candidates_b["visible_decoded"],
        "prompt_b_probability": candidates_b["probability"],
    })

    print("A:", prompt_a)
    print("B:", prompt_b)

    display(
        comparison.style.format({
            "prompt_a_probability": "{:.4%}",
            "prompt_b_probability": "{:.4%}",
        })
    )

    return comparison

### Sanity check

In [ ]:
warm_up_candidates = show_next_token_candidates(
    "Hello, how are",
    runtime,
    top_k=10,
)

### What to notice

Inspect both `token_piece` and `visible_decoded`.

A leading `␠` means the decoded token begins with a space. The token is therefore not identical to the same letters without that leading space. This is one reason **next token** is more precise than **next word**.


## 5. 🫵 Task 1 - Prediction Lineup

Before revealing any model output, predict one likely next token for each prompt.

Also predict how **concentrated** you expect the distribution to be:

- `high`: one continuation seems much more likely than the alternatives,
- `medium`: several continuations seem plausible,
- `low`: the prompt is open-ended or subjective.

Do not overthink token boundaries. Write the human-readable continuation you expect.


### Your predictions


Complete the `predicted_token` and `expected_concentration` fields before running the reveal cells.

Use `high`, `medium`, or `low` for the expected concentration. Your prediction does not need to match the tokenizer's exact whitespace or subword representation.

In [ ]:
prediction_lineup = pd.DataFrame([
    {
        "case": "Greeting",
        "prompt": "Hello, how are",
        "predicted_token": "",  # TODO
        "expected_concentration": "",  # TODO: high, medium, or low
    },
    {
        "case": "Factual completion",
        "prompt": "The capital of Israel is",
        "predicted_token": "",  # TODO
        "expected_concentration": "",  # TODO
    },
    {
        "case": "Action completion",
        "prompt": "To unlock the phone, she entered the",
        "predicted_token": "",  # TODO
        "expected_concentration": "",  # TODO
    },
    {
        "case": "Subjective completion",
        "prompt": "The best city to live in in Israel is",
        "predicted_token": "",  # TODO
        "expected_concentration": "",  # TODO
    },
])

prediction_lineup

### Reveal the model's candidates

In [ ]:
def normalize_prediction(text: str) -> str:
    return text.strip().casefold()


def prediction_rank(
    predicted_token: str,
    candidates: pd.DataFrame,
) -> int | None:
    normalized_prediction = normalize_prediction(predicted_token)

    for row in candidates.itertuples(index=False):
        if normalize_prediction(row.decoded) == normalized_prediction:
            return int(row.rank)

    return None


def detective_score(rank: int | None) -> int:
    if rank == 1:
        return 3
    if rank is not None and rank <= 3:
        return 2
    if rank is not None and rank <= 10:
        return 1
    return 0


def evaluate_prediction_lineup(
    prediction_table: pd.DataFrame,
    runtime: dict[str, Any],
) -> pd.DataFrame:
    results = []

    for row in prediction_table.itertuples(index=False):
        candidates = get_next_token_candidates(
            row.prompt,
            runtime,
            top_k=10,
        )

        rank = prediction_rank(row.predicted_token, candidates)
        top_probability = float(candidates.iloc[0]["probability"])
        second_probability = float(candidates.iloc[1]["probability"])

        results.append({
            "case": row.case,
            "prompt": row.prompt,
            "prediction": row.predicted_token,
            "expected_concentration": row.expected_concentration,
            "actual_top_token": candidates.iloc[0]["visible_decoded"],
            "top_probability": top_probability,
            "top1_top2_margin": top_probability - second_probability,
            "prediction_rank": rank,
            "score": detective_score(rank),
        })

    return pd.DataFrame(results)

In [ ]:
lineup_results = evaluate_prediction_lineup(
    prediction_lineup,
    runtime,
)

In [ ]:
lineup_results.style.format({
    "top_probability": "{:.4%}",
    "top1_top2_margin": "{:.4%}",
})

In [ ]:
print(
    "Your Token Detective score:",
    lineup_results["score"].sum(),
    "/",
    3 * len(lineup_results),
)

### Your analysis

After revealing the candidates, discuss:

- Which predictions appeared in the top ten?
- When the model disagreed, was its continuation still reasonable?
- Did token boundaries make any apparently obvious prediction harder to match?
- Which distribution was more or less concentrated than you expected?
- Why does a high next-token probability not prove factual correctness?

## 6. 🫵 Task 2 - Context Flips

For each case below:

1. Predict how the likely continuations will differ.
2. Run the supplied comparison.
3. Identify at least one candidate whose rank or probability changed substantially.
4. Explain what contextual clue caused the change.


### Case A - Same action, different object

In [ ]:
case_a = compare_prompt_pair(
    "She unlocked the door with the",
    "She unlocked the phone with the",
    runtime,
    top_k=6,
)

### Case B - Same ambiguous noun, different profession

In [ ]:
case_b = compare_prompt_pair(
    "The programmer found the bug in the",
    "The biologist found the bug in the",
    runtime,
    top_k=6,
)

### Case C - Reversing the relationship

In [ ]:
case_c = compare_prompt_pair(
    "The trophy would not fit in the suitcase because it was too",
    "The suitcase would not fit the trophy because it was too",
    runtime,
    top_k=6,
)

### Your analysis

For each case, identify at least one candidate whose rank or probability changed meaningfully. Explain the contextual clue that caused the shift.

Also note any case where the model did not produce the semantic change you expected. Treat that result as evidence about this particular model rather than correcting it to match your prediction.

## 7. 🫵 Task 3 - Create Your Own Case

Design a **minimal pair** of prompts:

- Keep most of the wording and grammatical structure unchanged.
- Change one word or one short phrase.
- Make a clear prediction about how the next-token distribution should change.
- Compare the top candidates.
- Write two or three sentences explaining the result.

Good cases often use an ambiguous word, a profession, a location, or a change in object type.


### Your case

Write two prompts that differ by only one word or one short phrase. Before running the comparison, predict how their candidate distributions should differ.

In [ ]:
custom_prompt_a = "TODO: write the first prompt"
custom_prompt_b = "TODO: write the minimally changed prompt"

In [ ]:
custom_case = compare_prompt_pair(
    custom_prompt_a,
    custom_prompt_b,
    runtime,
    top_k=8,
)

### Your interpretation

Write two or three sentences explaining:

- what change you expected,
- what actually changed in the candidate lists,
- whether the evidence supports your hypothesis.

## 8. 🫵 Optional Task 4 - A Probability-Concentration Clue

Compare a constrained factual completion with a subjective completion.

Predict which prompt will have:

- the larger top-token probability,
- the larger gap between the first and second candidates,
- the larger total probability assigned to its top five candidates.

These measurements describe the shape of one next-token distribution. They are **not calibrated confidence scores** and do not tell us whether a statement is true.


### Your implementation

In [ ]:
def distribution_clues(
    prompt: str,
    runtime: dict[str, Any],
    top_k: int = 5,
) -> dict[str, Any]:
    """
    Summarize several clues about the concentration of one next-token distribution.

    Return a dictionary containing:
    - prompt
    - top_token
    - top_probability
    - top1_top2_margin
    - top_<top_k>_probability_mass
    """

    # TODO:
    # 1. Obtain the candidates with get_next_token_candidates().
    # 2. Read the first and second candidate probabilities.
    # 3. Calculate the first-versus-second margin.
    # 4. Sum the probabilities of all returned candidates.
    # 5. Return the requested dictionary.
    raise NotImplementedError("Optional Task 4: implement distribution_clues")

In [ ]:
concentration_results = pd.DataFrame([
    distribution_clues(
        "The capital of Israel is",
        runtime,
    ),
    distribution_clues(
        "The best looking city in Israel is",
        runtime,
    ),
])

In [ ]:
concentration_results.style.format({
        "top_probability": "{:.4%}",
        "top1_top2_margin": "{:.4%}",
        "top_5_probability_mass": "{:.4%}",
    })

### Your analysis

Compare the factual and subjective prompts using the three measurements.

- Which distribution is more concentrated in your run?
- Do all three measurements support the same conclusion?
- Why are these values not calibrated confidence or factual-truth scores?

## 9. Reflection Questions

1. Which prompt produced the largest top-token probability? Was that also the prompt you considered easiest?
2. Where did a single context word cause the largest change?
3. Did the model ever produce a reasonable continuation you did not predict?
4. Why is inspecting the top ten candidates more informative than inspecting only the winner?
5. Why is next-token probability not the same thing as factual confidence?
6. How does this experiment explain why generated text can diverge after one different token choice?


### Your responses

Answer the reflection questions briefly, using evidence from the candidate tables produced during your own run.

## Takeaways

- An LLM produces a distribution over **tokens**, not complete thoughts or guaranteed words.
- Context changes which continuations are plausible and how probability is distributed among them.
- The highest-probability token is only one candidate at one generation step.
- Probability concentration is not factual verification.
- Small prompt or token changes can redirect the rest of a generated sequence.


<hr/>